In [1]:
# ── CELDA 1: Importaciones ────────────────────────────────────
import sys
import os

# Agrega la raíz del proyecto al path para importar scripts/
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Motor de base de datos reutilizando database.py del proyecto
from scripts.database import engine

# Estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Librerías importadas correctamente")
print(f"   pandas       {pd.__version__}")
print(f"   numpy        {np.__version__}")
print(f"   seaborn      {sns.__version__}")
print(f"   statsmodels  {sm.__version__}")

✅ Librerías importadas correctamente
   pandas       2.3.3
   numpy        2.4.3
   seaborn      0.13.2
   statsmodels  0.14.4


In [2]:
# ── CELDA 2: Carga de datos desde PostgreSQL (Supabase) ─────────────────

QUERY = """
SELECT
    id,
    nombre,
    magnitud_absoluta,
    diametro_min_km,
    diametro_max_km,
    es_peligroso,
    velocidad_km_s,
    distancia_km,
    fecha_aproximacion,
    fecha_extraccion
FROM asteroides
ORDER BY fecha_extraccion
"""

df = pd.read_sql(QUERY, engine)

print("✅ Datos cargados exitosamente desde Supabase")
print(f"   Filas    : {df.shape[0]:,}")
print(f"   Columnas : {df.shape[1]}")

print("\n📋 Primeras filas:")
display(df.head())

✅ Datos cargados exitosamente desde Supabase
   Filas    : 300
   Columnas : 10

📋 Primeras filas:


,id,nombre,magnitud_absoluta,diametro_min_km,diametro_max_km,es_peligroso,velocidad_km_s,distancia_km,fecha_aproximacion,fecha_extraccion
0,1,330659 (2008 GG2),23.01,0.066459,0.148607,False,7.190266,2.205655e+07,2026-04-11,2026-04-11 00:52:27.102419+00:00
1,2,363599 (2004 FG11),21.05,0.163891,0.366471,True,25.071287,8.454616e+06,2026-04-11,2026-04-11 00:52:27.102558+00:00
2,3,(2017 GS6),24.80,0.029144,0.065169,False,17.306900,7.146886e+07,2026-04-11,2026-04-11 00:52:27.102571+00:00
3,4,(2017 UR5),28.70,0.004837,0.010815,False,8.500920,5.696653e+07,2026-04-11,2026-04-11 00:52:27.102580+00:00
4,5,(2019 GK3),29.50,0.003346,0.007482,False,5.538608,1.289511e+07,2026-04-11,2026-04-11 00:52:27.102587+00:00


In [4]:
# ── CELDA 3: Validación y Exploración ────────────────────────────────────────
print("=" * 55)
print("VALIDACIÓN DEL DATASET - NASA ASTEROIDS")
print("=" * 55)

# Tipos y nulos
print("\n📋 Tipos de datos y valores nulos:")
print(df.dtypes.to_frame('dtype').join(
    df.isnull().sum().to_frame('nulos')
))

# Balance de clases
# Nota: es_peligroso suele venir como booleano (True/False) o int (1/0)
balance = df['es_peligroso'].value_counts()
pct     = df['es_peligroso'].value_counts(normalize=True) * 100

print(f"\n🎯 Balance de la variable objetivo (es_peligroso):")
# Manejamos el acceso a las etiquetas de forma segura (sea 1/0 o True/False)
for val in balance.index:
    label = "Peligroso" if val in [1, True] else "No Peligroso"
    print(f"   {label:<15} ({val}): {balance[val]:>5,}  ({pct[val]:.1f}%)")

# Exploración de magnitudes
print(f"\n🔭 Estadísticas de Magnitud Absoluta:")
print(df['magnitud_absoluta'].describe().to_frame().T.to_string())

# Verificación de fechas
print(f"\n📅 Rango de extracción de datos:")
print(f"   Inicio: {df['fecha_extraccion'].min()}")
print(f"   Fin   : {df['fecha_extraccion'].max()}")

VALIDACIÓN DEL DATASET - NASA ASTEROIDS

📋 Tipos de datos y valores nulos:
                                  dtype  nulos
id                                int64      0
nombre                           object      0
magnitud_absoluta               float64      0
diametro_min_km                 float64      0
diametro_max_km                 float64      0
es_peligroso                       bool      0
velocidad_km_s                  float64      0
distancia_km                    float64      0
fecha_aproximacion               object      0
fecha_extraccion    datetime64[ns, UTC]      0

🎯 Balance de la variable objetivo (es_peligroso):
   No Peligroso    (False):   281  (93.7%)
   Peligroso       (True):    19  (6.3%)

🔭 Estadísticas de Magnitud Absoluta:
                   count       mean       std    min     25%   50%       75%    max
magnitud_absoluta  300.0  25.076513  2.370861  17.41  23.775  25.4  26.72925  30.35

📅 Rango de extracción de datos:
   Inicio: 2026-04-11 00:52:27.102

In [8]:
# ── CELDA 4: Preprocesamiento y Balanceo de Clases ───────────────────────────
df_prep = df.copy()

# 4.1 Ingeniería de Variables (Ajustada)
# Aplicamos raíz cuadrada al diámetro máximo como se solicitó
df_prep['raiz_diametro'] = np.sqrt(df_prep['diametro_max_km']) 

# Brillo relativo: magnitud_absoluta multiplicada por -1 y aplicada escala logarítmica
df_prep['brillo_relativo'] = df_prep['magnitud_absoluta'] * -1
df_prep['brillo_log'] = np.log1p(df_prep['brillo_relativo'].abs())

# 4.2 Relleno de nulos con mediana en variables numéricas
num_cols = ['magnitud_absoluta', 'diametro_min_km', 'diametro_max_km', 'velocidad_km_s', 'distancia_km']
for col in num_cols:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())

# 4.3 Definir X e y antes del balanceo
# Utilizamos las variables transformadas y las cinemáticas de la NASA
features = ['brillo_log', 'raiz_diametro', 'velocidad_km_s', 'distancia_km']
X_unbalanced = df_prep[features]
y_unbalanced = df_prep['es_peligroso'].astype(int)

# 4.4 BALANCEO DE CLASES: Oversampling Manual (Sin SMOTE)
from sklearn.utils import resample

# Unimos temporalmente para asegurar que el remuestreo sea consistente
df_to_resample = pd.concat([X_unbalanced, y_unbalanced], axis=1)

# Separar clases mayoritaria y minoritaria
df_maj = df_to_resample[df_to_resample.es_peligroso == 0]
df_min = df_to_resample[df_to_resample.es_peligroso == 1]

# Sobremuestreo de la clase minoritaria (Peligrosos) hasta alcanzar el tamaño de la mayoritaria
df_min_upsampled = resample(df_min, 
                            replace=True,     # Permite duplicar registros
                            n_samples=len(df_maj), # Igualar al conteo de la clase 0
                            random_state=42)

# Recombinar el dataset balanceado
df_final = pd.concat([df_maj, df_min_upsampled])

X = df_final.drop(columns=['es_peligroso'])
y = df_final['es_peligroso']

print("✅ Preprocesamiento y Oversampling completado con variable 'raiz_diametro'")
print(f"   Shape X final: {X.shape}")
print(f"   Variables finales: {X.columns.tolist()}")
print(f"\n🎯 Balance final de clases:")
print(f"   No Peligrosos (0): {(y==0).sum():,} ({ (y==0).mean()*100 :.1f}%)")
print(f"   Peligrosos    (1): {(y==1).sum():,} ({ (y==1).mean()*100 :.1f}%)")

✅ Preprocesamiento y Oversampling completado con variable 'raiz_diametro'
   Shape X final: (562, 4)
   Variables finales: ['brillo_log', 'raiz_diametro', 'velocidad_km_s', 'distancia_km']

🎯 Balance final de clases:
   No Peligrosos (0): 281 (50.0%)
   Peligrosos    (1): 281 (50.0%)


In [12]:
# ── CELDA 5: Función de entrenamiento + evaluación (NASA Version) ──────────────
from sklearn.tree import DecisionTreeClassifier  # <--- ESTA ES LA LÍNEA QUE FALTA
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

def entrenar_evaluar(X, y, test_size, random_state=42, nombre="Split", max_depth=None):
    """
    Entrena DecisionTreeClassifier con un split train/test y devuelve
    un dict con métricas + datos para curva ROC y matriz de confusión.
    """
    # Dividir datos manteniendo la proporción de la variable objetivo
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Inicializar el modelo CART
    modelo = DecisionTreeClassifier(
        criterion='gini',
        max_depth=max_depth,
        min_samples_split=10, 
        min_samples_leaf=5,   
        random_state=random_state
    )
    
    # Entrenamiento
    modelo.fit(X_train, y_train)

    # Predicciones
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    # Datos para Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    return {
        'nombre'    : nombre,
        'train_size': len(X_train),
        'test_size' : len(X_test),
        'accuracy'  : accuracy_score(y_test, y_pred),
        'precision' : precision_score(y_test, y_pred, zero_division=0),
        'recall'    : recall_score(y_test, y_pred, zero_division=0),
        'f1'        : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'   : roc_auc_score(y_test, y_prob),
        'fpr'       : fpr,
        'tpr'       : tpr,
        'y_test'    : y_test,
        'y_pred'    : y_pred,
        'modelo'    : modelo,
    }

print("✅ Función `entrenar_evaluar` actualizada con las importaciones necesarias")

✅ Función `entrenar_evaluar` actualizada con las importaciones necesarias


In [13]:
# ── CELDA 6: Ejecutar los tres splits (Evaluación de Consistencia) ─────────────

# Ejecutamos la función definida en la Celda 5 para los tres escenarios
# Nota: Puedes pasar un valor a `max_depth` si ya identificaste uno óptimo, 
# de lo contrario, por ahora lo dejamos en None o un valor preventivo como 10.
resultados = [
    entrenar_evaluar(X, y, test_size=0.20, nombre="Split 80/20", max_depth=10),
    entrenar_evaluar(X, y, test_size=0.40, nombre="Split 60/40", max_depth=10),
    entrenar_evaluar(X, y, test_size=0.30, nombre="Split 70/30", max_depth=10),
]

# ── Imprimir métricas en consola ──────────────────────────────────────────────
print("=" * 65)
print(f"{'MÉTRICAS COMPARATIVAS NASA — ÁRBOL CART':^65}")
print("=" * 65)
header = f"{'Split':<12} {'N Train':>8} {'N Test':>8} {'Acc':>7} {'Prec':>7} {'Recall':>7} {'F1':>7} {'AUC':>7}"
print(header)
print("-" * 65)

for r in resultados:
    print(
        f"{r['nombre']:<12} {r['train_size']:>8,} {r['test_size']:>8,}"
        f" {r['accuracy']:>7.4f} {r['precision']:>7.4f}"
        f" {r['recall']:>7.4f} {r['f1']:>7.4f} {r['roc_auc']:>7.4f}"
    )
print("=" * 65)
print("⚠️  Nota: El Recall es crítico; mide qué porcentaje de peligrosos reales detectamos.")

             MÉTRICAS COMPARATIVAS NASA — ÁRBOL CART             
Split         N Train   N Test     Acc    Prec  Recall      F1     AUC
-----------------------------------------------------------------
Split 80/20       449      113  1.0000  1.0000  1.0000  1.0000  1.0000
Split 60/40       337      225  0.9867  0.9739  1.0000  0.9868  0.9947
Split 70/30       393      169  0.9941  0.9882  1.0000  0.9941  0.9995
⚠️  Nota: El Recall es crítico; mide qué porcentaje de peligrosos reales detectamos.
